# Bactomata // Data Analysis

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import os

from matplotlib.colors import to_rgb
from matplotlib.patches import Patch

## User-defined parameters

The main user-defined parameter is `expeID`, which identifies the experiment being analyzed. This name is used to locate the experiment folder, build the expected input and output file names, and label the parsed data object. To analyze a different experiment, update `expeID` so it matches the folder and filenames for that experiment.

In [2]:
expeID = "Bactomata_Example"
data_file="DO_kinetic_48h_rhizobia_SynComs_9_strains_3_replicates_25042026.txt"
plate_id='Plate1'

Define the project root in Google Drive and change the working directory so all later paths can be written relative to the Bactomata folder.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
dirName='/content/drive/Shareddrives/ESB_Code/Bactomata/'
PROJECT_ROOT = Path(dirName)

os.chdir(PROJECT_ROOT)
print('Current directory:', Path.cwd())

Current directory: /content/drive/Shareddrives/ESB_Code/Bactomata


## Project and experiment file structure

Bactomata uses a project-level folder that contains the reusable source code, notebooks, and one folder per experiment. Each experiment is stored under `experiments/<expeID>/`, so the raw inputs, parsed data, exported tables, figures, and notes for a given experiment remain together.

A typical project structure is:

```text
Bactomata/
├── bactomata/
│   ├── __init__.py
│   ├── parser.py
│   ├── processing.py
│   └── plotter.py
│
├── notebooks/
│   └── Bactomata_DataLoader_Example.ipynb
│
└── experiments/
    └── <expeID>/
        ├── raw/
        │   ├── plate_reader/
        │   │   └── <BioTek_export_file>.txt
        │   │
        │   └── layouts/
        │       ├── Bactomata_<expeID>_media_layout.txt
        │       ├── Bactomata_<expeID>_bacteria_layout.txt
        │       ├── Bactomata_<expeID>_media_key_dict.txt
        │       ├── Bactomata_<expeID>_bacteria_key_dict.txt
        │       └── Bactomata_<expeID>_trough_layout.txt
        │
        ├── processed/
        │   ├── Bactomata_<expeID>_parsed.pkl
        │   ├── Bactomata_<expeID>_annotated_data.csv
        │   └── Bactomata_<expeID>_well_annotations.csv
        │
        ├── figures/
        └── notes/

The experiment identifier is used to define the folder structure for the current experiment. These paths point to the raw input files, layout files, processed outputs, figures, and notes. The output folders are created automatically if they do not already exist.

In [5]:

EXPERIMENT_DIR = Path("experiments") / expeID

RAW_DIR = EXPERIMENT_DIR / "raw"
LAYOUT_DIR = RAW_DIR / "layouts"
PLATE_READER_DIR = RAW_DIR / "plate_reader"
PROCESSED_DIR = EXPERIMENT_DIR / "processed"
FIGURES_DIR = EXPERIMENT_DIR / "figures"
NOTES_DIR = EXPERIMENT_DIR / "notes"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
NOTES_DIR.mkdir(parents=True, exist_ok=True)

The input files for the current experiment are defined from the experiment-specific folders. These include the raw BioTek export, the media and bacteria layouts, the corresponding key dictionaries, and the trough layout used during plate setup.

In [6]:
plate_reader_path = PLATE_READER_DIR / data_file

media_layout_path = LAYOUT_DIR / f"{expeID}_media_layout.txt"
bacteria_layout_path = LAYOUT_DIR / f"{expeID}_bacteria_layout.txt"
media_key_dict_path = LAYOUT_DIR / f"{expeID}_media_key_dict.txt"
bacteria_key_dict_path = LAYOUT_DIR / f"{expeID}_bacteria_key_dict.txt"
trough_layout_path = LAYOUT_DIR / f"{expeID}_trough_layout.txt"

Import the processing and plotting modules. During development, the modules are reloaded so that changes made to the `.py` source files are available in the current notebook session.


In [ ]:
import importlib
import bactomata.processing as bp
import bactomata.plotter as bplt
import bactomata.parser as bprs

importlib.reload(bp)
importlib.reload(bplt)
importlib.reload(bprs)


# BioTek Data Loading


The function `build_experiment_object()` combines the raw BioTek export with the media layout, bacteria layout, key dictionaries, and trough layout. The output is a structured experiment object in which each measurement is linked to its well position, media condition, and bacteria condition.

In [15]:
experiment = bprs.build_experiment_object(
    expeID=expeID,
    plate_reader_path=plate_reader_path,
    media_layout_path=media_layout_path,
    bacteria_layout_path=bacteria_layout_path,
    media_key_dict_path=media_key_dict_path,
    bacteria_key_dict_path=bacteria_key_dict_path,
    trough_layout_path=trough_layout_path,
    plate_id=plate_id,
)

FileNotFoundError: [Errno 2] No such file or directory: 'experiments/Bactomata_Example/raw/layouts/Bactomata_Bactomata_Example_media_layout.txt'

The parsed experiment is saved in the processed folder for the current experiment. The PKL file stores the complete Bactomata object for later analysis, while the exported tables provide human-readable versions of the annotated data and well annotations.

In [ ]:
# Output paths for this experiment
PROCESSED_DIR = EXPERIMENT_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_pkl_path = PROCESSED_DIR / f"Bactomata_{expeID}_parsed.pkl"
output_tables_dir = PROCESSED_DIR

bprs.save_experiment_pkl(experiment, output_pkl_path)
bprs.export_experiment_tables(experiment, output_tables_dir)

print("Saved PKL to:", output_pkl_path)
print("Saved tables to:", output_tables_dir)

## Bactomata Data Processing

The variable `experiment` is the central data structure produced by the Bactomata parser. It is a dictionary that stores the parsed experiment in a standardized format:

- `experiment["data"]`: annotated measurement table, with one row per well, time point, and signal.
- `experiment["well_annotations"]`: plate map linking each well to its media condition and bacteria condition.
- `experiment["layouts"]`: original media, bacteria, and trough layouts.
- `experiment["key_dicts"]`: definitions of the media and bacteria keys used in the layouts.
- `experiment["metadata"]`: basic information extracted from the plate-reader file.
- `experiment["warnings"]`: validation warnings generated during parsing.
- `experiment["extra"]`: optional space for adding derived variables or later analyses.

In this section, we use this object to extract the measurements associated with selected media and bacteria conditions, inspect replicate wells, reshape the data into different formats, and calculate simple summaries across replicates.

### Load a previously parsed experiment

The function `load_experiment_pkl()` loads an existing Bactomata experiment object from a PKL file. This allows downstream processing and plotting to start from a previously saved parsed experiment.

In [ ]:
bprs.load_experiment_pkl(output_pkl_path)

### Extract time-series data for one condition

The function `get_growth_data()` extracts the time-series data for a selected media and bacteria condition. The returned table keeps replicate wells as separate observations.

In [ ]:
growth = bp.get_growth_data(
    experiment,
    media_key="M1",
    bacteria_key="B1",
    signal="OD630",
)

growth.head()

List the replicate wells associated with the selected media and bacteria condition.


In [ ]:
growth["well"].unique()

Count the number of replicate wells for the selected condition.


In [ ]:
growth["well"].nunique()

Convert the selected growth data into a matrix with one column per replicate well. This format is useful for quick inspection and some downstream analyses.


In [ ]:
growth_matrix = bp.get_growth_matrix(
    experiment,
    media_key="M1",
    bacteria_key="B1",
    signal="OD630",
)

growth_matrix.head()

Summarize the selected condition by calculating the mean, standard deviation, and number of replicate wells at each time point.


In [ ]:
summary = bp.summarize_growth_data(
    experiment,
    media_key="M1",
    bacteria_key="B1",
    signal="OD630",
)

summary.head()

# Data Plotting

Once the parsed experiment has been loaded, the annotated data can be queried directly by media condition, bacteria condition, signal, or well position. The examples below illustrate the basic plotting workflow: inspecting individual replicate curves, calculating mean time series across replicate wells, comparing conditions, and visualizing the full plate layout for quality control.

### Inspect the full plate layout

The function `plot_plate_overview()` displays the full plate in its original spatial arrangement. Each well is shown as a small subplot, using the selected signal and plotting mode. The function `make_condition_color_map()` assigns colors to each media and bacteria combination for consistent visual annotation.

In [ ]:
condition_color_map = bplt.make_condition_color_map(experiment)

fig, axes = bplt.plot_plate_overview(
    experiment,
    signal="OD630",
    mode="time_series",
    condition_color_map=condition_color_map,
    show_well_labels=True,
    show_condition_legend=True,
    cell_size=1.3,
)

plt.show()

### Inspect a single well

The function `plot_well_data()` plots the time-series data from one selected well. The well is specified through the `well` argument.

In [ ]:
bplt.plot_well_data(
    experiment,
    well="A3",
    signal="OD630",
)

plt.tight_layout()
plt.show()

### Inspect selected wells together

The function `plot_wells()` plots the time-series data from a user-defined list of wells in the same figure. The wells are passed explicitly through the `wells` argument.

In [ ]:
bplt.plot_wells(
    experiment,
    wells=["A2", "B2", "C2"],
    signal="OD630",
    fill_area=False,
)

plt.tight_layout()
plt.show()

### Inspect all wells for one condition

The wells for a selected media and bacteria condition can be retrieved with `get_wells_for_condition()` and passed to `plot_wells()`.

In [ ]:
this_wells = bplt.get_wells_for_condition(
    experiment,
    media_key="M1",
    bacteria_key="B1",
)
print(this_wells)


In [ ]:
bplt.plot_wells(
    experiment,
    wells=this_wells,
    signal="OD630",
    fill_area=False,
)

plt.tight_layout()
plt.show()



The same plot can also be generated directly with `plot_condition_wells()`, which uses the media and bacteria keys as input instead of an explicit well list.

In [ ]:
bplt.plot_condition_wells(
    experiment,
    media_key="M1",
    bacteria_key="B1",
    signal="OD630",
)

plt.tight_layout()
plt.show()

### Plot the mean time series for one condition

The function `summarize_growth_data()` calculates the mean over replicate wells at each time point for a selected media and bacteria condition. The resulting summary table is then passed to `plot_summary_curve()` to plot the mean response as a single curve, with optional variation across replicates shown around the mean.

In [ ]:
summary = bp.summarize_growth_data(
    experiment,
    media_key="M1",
    bacteria_key="B1",
    signal="OD630",
)

bplt.plot_summary_curve(
    summary,
    signal="OD630",
    title="M1 + B1",
    label="B1"
)

plt.tight_layout()
plt.show()

### Compare multiple bacteria conditions within one medium

The function `plot_bacteria_comparison()` compares selected bacteria conditions within one media condition. For each bacteria condition, replicate wells are summarized over time and plotted together in a single figure.

In [ ]:
bplt.plot_bacteria_comparison(
    experiment,
    media_key="M1",
    bacteria_keys=["B1", "B2", "B3", "B4"],
    signal="OD630",
)

plt.tight_layout()
plt.show()

### Compare one bacteria condition across media

The function `plot_media_comparison()` compares one bacteria condition across selected media conditions. For each media condition, replicate wells are summarized over time and plotted together in a single figure.

In [ ]:
bplt.plot_media_comparison(
    experiment,
    media_keys=["M1", "M2"],
    bacteria_key="B1",
    signal="OD630",
)

plt.tight_layout()
plt.show()

Compare the same bacteria key across two or more media conditions.


In [ ]:
conditions = bp.list_conditions(experiment)
conditions

### Compare all bacteria conditions within one medium

The function `plot_all_bacteria_in_media()` plots all bacteria conditions present within a selected media condition. For each bacteria condition, replicate wells are summarized over time and plotted in the same figure.

In [ ]:
bplt.plot_all_bacteria_in_media(
    experiment,
    media_key="M1",
    signal="OD630",
)

plt.tight_layout()
plt.show()

### Compare one bacteria condition between two media

The function `compare_two_media()` compares one bacteria condition between two selected media conditions. The two media keys are passed as `media1` and `media2`, and the bacteria condition is specified with `bacteria_key`.

In [ ]:
bplt.compare_two_media(
    experiment,
    media1="M1",
    media2="M2",
    bacteria_key="B1",
    signal="OD630",
)

plt.tight_layout()
plt.show()

This is equivalent to using the more general function:

In [ ]:
bplt.plot_media_comparison(
    experiment,
    media_keys=["M1", "M2"],
    bacteria_key="B1",
    signal="OD630",
)

plt.tight_layout()
plt.show()